In [1]:
# Cell 1 — Load verified data
import pandas as pd
import numpy as np

master = pd.read_csv("../data/master_labeled.csv")

# Drop economy column if present
if "economy" in master.columns:
    master = master.drop(columns=["economy"])

print(master.shape)
print(master.columns.tolist())
print(master.head())

(30, 16)
['year', 'gdp_growth', 'inflation_yoy', 'forex_import_cover', 'remittance_pct_gdp', 'remittance_growth_yoy', 'india_gdp_growth', 'india_inflation', 'current_account_pct_gdp', 't1_gdp', 't2_forex', 't3_inflation', 't4_remittance', 't5_current_account', 'triggers_fired', 'crisis']
   year  gdp_growth  inflation_yoy  forex_import_cover  remittance_pct_gdp  \
0  1995    3.468452       7.622970            4.669444            1.291100   
1  1996    5.328284       9.220467            4.257226            0.976652   
2  1997    5.048613       4.009989            4.137290            1.005512   
3  1998    3.016389      11.244468            6.569615            1.390061   
4  1999    4.412573       7.451113            6.151447            1.658099   

   remittance_growth_yoy  india_gdp_growth  india_inflation  \
0                    NaN          7.574492        10.224886   
1             -22.284313          7.549522         8.977152   
2              11.997094          4.049821         7.

In [2]:
# Cell 2 — Separate raw indicators from labels
# We engineer features only from raw indicators
# Triggers and labels stay separate, never used as features

raw_cols = [
    'year',
    'gdp_growth',
    'inflation_yoy',
    'forex_import_cover',
    'remittance_pct_gdp',
    'remittance_growth_yoy',
    'india_gdp_growth',
    'india_inflation',
    'current_account_pct_gdp'
]

label_cols = [
    'year',
    'crisis',
    'triggers_fired',
    't1_gdp', 't2_forex', 't3_inflation', 't4_remittance', 't5_current_account'
]

raw = master[raw_cols].copy()
labels = master[label_cols].copy()

print("Raw shape:", raw.shape)
print("Labels shape:", labels.shape)
print("\nRaw columns:", raw.columns.tolist())

Raw shape: (30, 9)
Labels shape: (30, 8)

Raw columns: ['year', 'gdp_growth', 'inflation_yoy', 'forex_import_cover', 'remittance_pct_gdp', 'remittance_growth_yoy', 'india_gdp_growth', 'india_inflation', 'current_account_pct_gdp']


In [3]:
# Cell 3 — t-1 lag features
# Rule: every feature predicting year T uses year T-1 value only
# This is the single most important leakage prevention step

features = pd.DataFrame()
features['year'] = raw['year']

# Priority features — t-1 lags
features['gdp_growth_lag1']              = raw['gdp_growth'].shift(1)
features['inflation_lag1']               = raw['inflation_yoy'].shift(1)
features['forex_import_cover_lag1']      = raw['forex_import_cover'].shift(1)
features['remittance_pct_gdp_lag1']      = raw['remittance_pct_gdp'].shift(1)
features['remittance_growth_lag1']       = raw['remittance_growth_yoy'].shift(1)
features['india_gdp_growth_lag1']        = raw['india_gdp_growth'].shift(1)
features['india_gdp_growth_lag2']        = raw['india_gdp_growth'].shift(2)
features['india_inflation_lag1']         = raw['india_inflation'].shift(1)
features['current_account_lag1']         = raw['current_account_pct_gdp'].shift(1)

print(features.shape)
print(features.head(5).to_string())

(30, 10)
   year  gdp_growth_lag1  inflation_lag1  forex_import_cover_lag1  remittance_pct_gdp_lag1  remittance_growth_lag1  india_gdp_growth_lag1  india_gdp_growth_lag2  india_inflation_lag1  current_account_lag1
0  1995              NaN             NaN                      NaN                      NaN                     NaN                    NaN                    NaN                   NaN                   NaN
1  1996         3.468452        7.622970                 4.669444                 1.291100                     NaN               7.574492                    NaN             10.224886             -8.097741
2  1997         5.328284        9.220467                 4.257226                 0.976652              -22.284313               7.549522               7.574492              8.977152             -7.223150
3  1998         5.048613        4.009989                 4.137290                 1.005512               11.997094               4.049821               7.549522           

In [4]:
# Cell 4 — Rolling mean and std features
# 3-year rolling window on key indicators
# .shift(1) ensures window ends at T-1, not T (no leakage)

# GDP 3-year rolling std — captures instability
features['gdp_growth_roll3_std']  = (
    raw['gdp_growth'].shift(1).rolling(3).std()
)

# GDP 3-year rolling mean — captures medium trend
features['gdp_growth_roll3_mean'] = (
    raw['gdp_growth'].shift(1).rolling(3).mean()
)

# Forex 3-year rolling mean — captures reserve trend
features['forex_roll3_mean'] = (
    raw['forex_import_cover'].shift(1).rolling(3).mean()
)

# Inflation 3-year rolling mean — captures inflation persistence
features['inflation_roll3_mean'] = (
    raw['inflation_yoy'].shift(1).rolling(3).mean()
)

# Remittance pct GDP 3-year rolling mean
features['remittance_roll3_mean'] = (
    raw['remittance_pct_gdp'].shift(1).rolling(3).mean()
)

print(features.shape)
print(features[['year','gdp_growth_roll3_std','gdp_growth_roll3_mean',
                'forex_roll3_mean','inflation_roll3_mean']].head(10).to_string())

(30, 15)
   year  gdp_growth_roll3_std  gdp_growth_roll3_mean  forex_roll3_mean  inflation_roll3_mean
0  1995                   NaN                    NaN               NaN                   NaN
1  1996                   NaN                    NaN               NaN                   NaN
2  1997                   NaN                    NaN               NaN                   NaN
3  1998              1.002838               4.615116          4.354653              6.951142
4  1999              1.261811               4.464429          4.988044              8.158308
5  2000              1.039536               4.159192          5.619451              7.568523
6  2001              1.595807               4.542988          6.404112              7.058134
7  2002              0.940320               5.137488          6.669771              4.206079
8  2003              3.183944               3.706678          7.088452              2.732174
9  2004              2.492009               2.955024         

In [5]:
# Cell 5 — Nepal-India inflation spread (F14)
# Spread = Nepal inflation minus India inflation, lagged 1 year
# Captures real exchange rate pressure under the INR peg
# Positive spread = Nepal inflating faster than India = competitiveness loss

features['nepal_india_inflation_spread_lag1'] = (
    raw['inflation_yoy'].shift(1) - raw['india_inflation'].shift(1)
)

print(features[['year','nepal_india_inflation_spread_lag1']].head(10).to_string())

   year  nepal_india_inflation_spread_lag1
0  1995                                NaN
1  1996                          -2.601917
2  1997                           0.243314
3  1998                          -3.154264
4  1999                          -1.986371
5  2000                           2.781292
6  2001                          -1.530616
7  2002                          -1.090989
8  2003                          -1.267753
9  2004                           1.901150


In [6]:
# Cell 6 — Forex rapid decline flag
# Binary feature: fires when forex dropped more than 2 months
# in the prior year — captures sudden reserve stress
# Uses T-1 to T-2 change, so no leakage into year T

forex_change = raw['forex_import_cover'].shift(1) - raw['forex_import_cover'].shift(2)
features['forex_rapid_decline_lag1'] = (forex_change < -2).astype(int)

# Verify — show years where flag fired
fired = features[features['forex_rapid_decline_lag1'] == 1][['year','forex_rapid_decline_lag1']]
print("Forex rapid decline flag fired in:")
print(fired.to_string())

Forex rapid decline flag fired in:
    year  forex_rapid_decline_lag1
22  2017                         1
24  2019                         1
27  2022                         1


In [7]:
# Cell 7 — Merge features with crisis labels
modeling_df = features.merge(
    labels[['year', 'crisis', 'triggers_fired']],
    on='year'
)

print(modeling_df.shape)
print(modeling_df.columns.tolist())
print(modeling_df.head(5).to_string())

(30, 19)
['year', 'gdp_growth_lag1', 'inflation_lag1', 'forex_import_cover_lag1', 'remittance_pct_gdp_lag1', 'remittance_growth_lag1', 'india_gdp_growth_lag1', 'india_gdp_growth_lag2', 'india_inflation_lag1', 'current_account_lag1', 'gdp_growth_roll3_std', 'gdp_growth_roll3_mean', 'forex_roll3_mean', 'inflation_roll3_mean', 'remittance_roll3_mean', 'nepal_india_inflation_spread_lag1', 'forex_rapid_decline_lag1', 'crisis', 'triggers_fired']
   year  gdp_growth_lag1  inflation_lag1  forex_import_cover_lag1  remittance_pct_gdp_lag1  remittance_growth_lag1  india_gdp_growth_lag1  india_gdp_growth_lag2  india_inflation_lag1  current_account_lag1  gdp_growth_roll3_std  gdp_growth_roll3_mean  forex_roll3_mean  inflation_roll3_mean  remittance_roll3_mean  nepal_india_inflation_spread_lag1  forex_rapid_decline_lag1  crisis  triggers_fired
0  1995              NaN             NaN                      NaN                      NaN                     NaN                    NaN                    N

In [8]:
# Cell 8 — NaN audit
# We need to know exactly how many rows survive for modeling
# and which features are causing the most data loss

print("NaN count per column:")
print(modeling_df.isnull().sum().to_string())
print()

# Rows with no NaN in any feature column
feature_cols = [c for c in modeling_df.columns 
                if c not in ['year','crisis','triggers_fired']]
clean_rows = modeling_df.dropna(subset=feature_cols)
print(f"Total rows: {len(modeling_df)}")
print(f"Rows with no NaN: {len(clean_rows)}")
print(f"Years lost: {modeling_df[~modeling_df['year'].isin(clean_rows['year'])]['year'].tolist()}")

NaN count per column:
year                                 0
gdp_growth_lag1                      1
inflation_lag1                       1
forex_import_cover_lag1              1
remittance_pct_gdp_lag1              1
remittance_growth_lag1               3
india_gdp_growth_lag1                1
india_gdp_growth_lag2                2
india_inflation_lag1                 1
current_account_lag1                 1
gdp_growth_roll3_std                 3
gdp_growth_roll3_mean                3
forex_roll3_mean                     3
inflation_roll3_mean                 3
remittance_roll3_mean                3
nepal_india_inflation_spread_lag1    1
forex_rapid_decline_lag1             0
crisis                               0
triggers_fired                       0

Total rows: 30
Rows with no NaN: 26
Years lost: [1995, 1996, 1997, 2003]


In [9]:
# Cell 9 — Drop NaN rows and save final feature set
# Decision: keep remittance_growth_lag1, accept 26 usable rows
# 2003 remittance growth cannot be recovered — NRB structural break
# in 2002 base value makes the growth rate unreliable
# Years lost: 1995, 1996, 1997 (rolling window), 2003 (remittance NaN)

feature_cols = [c for c in modeling_df.columns
                if c not in ['year', 'crisis', 'triggers_fired']]

final_df = modeling_df.dropna(subset=feature_cols).reset_index(drop=True)

print(f"Final modeling dataset: {final_df.shape}")
print(f"Years covered: {final_df['year'].min()} - {final_df['year'].max()}")
print(f"Crisis years in final set: {final_df[final_df['crisis']==1]['year'].tolist()}")
print(f"Crisis rate: {final_df['crisis'].mean()*100:.1f}%")

Final modeling dataset: (26, 19)
Years covered: 1998 - 2024
Crisis years in final set: [1998, 2008, 2009, 2010, 2011, 2016, 2021]
Crisis rate: 26.9%


In [10]:
# Cell 10 — Save final feature set
# 26 rows, 1998-2024, 7 crisis years, 26.9% crisis rate
# 1996 crisis year lost due to rolling window minimum (expected)
# 2003 lost due to remittance structural break (documented)

final_df.to_csv("../data/features.csv", index=False)

print("Saved: ../data/features.csv")
print(f"Shape: {final_df.shape}")
print(f"Feature columns: {[c for c in final_df.columns if c not in ['year','crisis','triggers_fired']]}")

Saved: ../data/features.csv
Shape: (26, 19)
Feature columns: ['gdp_growth_lag1', 'inflation_lag1', 'forex_import_cover_lag1', 'remittance_pct_gdp_lag1', 'remittance_growth_lag1', 'india_gdp_growth_lag1', 'india_gdp_growth_lag2', 'india_inflation_lag1', 'current_account_lag1', 'gdp_growth_roll3_std', 'gdp_growth_roll3_mean', 'forex_roll3_mean', 'inflation_roll3_mean', 'remittance_roll3_mean', 'nepal_india_inflation_spread_lag1', 'forex_rapid_decline_lag1']


In [11]:
# Cell 11 — Correlation matrix
# Check correlations between features and with crisis label
# Features with correlation > 0.85 with each other are candidates for removal
# Keep the one with higher correlation to crisis label

feature_cols = [c for c in final_df.columns 
                if c not in ['year', 'crisis', 'triggers_fired']]

X = final_df[feature_cols]
y = final_df['crisis']

# Correlation with crisis label
crisis_corr = X.corrwith(y).sort_values(ascending=False)
print("Correlation with crisis label:")
print(crisis_corr.round(3).to_string())

print()

# Inter-feature correlation matrix
corr_matrix = X.corr().round(2)
print("High inter-feature correlations (>0.85):")
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        val = corr_matrix.iloc[i,j]
        if abs(val) > 0.85:
            high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], val))

if high_corr:
    for a, b, v in high_corr:
        print(f"  {a} <-> {b}: {v}")
else:
    print("  None found above 0.85")

Correlation with crisis label:
india_inflation_lag1                 0.350
current_account_lag1                 0.209
inflation_roll3_mean                 0.120
remittance_pct_gdp_lag1              0.115
remittance_roll3_mean                0.088
inflation_lag1                       0.076
india_gdp_growth_lag2                0.061
gdp_growth_roll3_mean                0.059
remittance_growth_lag1               0.035
forex_import_cover_lag1              0.015
gdp_growth_roll3_std                -0.100
forex_roll3_mean                    -0.173
gdp_growth_lag1                     -0.217
forex_rapid_decline_lag1            -0.219
india_gdp_growth_lag1               -0.322
nepal_india_inflation_spread_lag1   -0.354

High inter-feature correlations (>0.85):
  remittance_pct_gdp_lag1 <-> remittance_roll3_mean: 0.99


In [12]:
# Cell 12 — Drop redundant and weak features
# Dropped: remittance_roll3_mean (0.99 corr with remittance_pct_gdp_lag1, weaker signal)
# Dropped: forex_import_cover_lag1 (corr=0.015, forex_roll3_mean captures trend better)
# Dropped: india_gdp_growth_lag2 (corr=0.061, lag1 is stronger at -0.322)
# Dropped: remittance_growth_lag1 (corr=0.035, near zero signal)

drop_cols = [
    'remittance_roll3_mean',
    'forex_import_cover_lag1',
    'india_gdp_growth_lag2',
    'remittance_growth_lag1'
]

final_df = final_df.drop(columns=drop_cols)

feature_cols = [c for c in final_df.columns
                if c not in ['year', 'crisis', 'triggers_fired']]

print(f"Features remaining: {len(feature_cols)}")
print(feature_cols)

Features remaining: 12
['gdp_growth_lag1', 'inflation_lag1', 'remittance_pct_gdp_lag1', 'india_gdp_growth_lag1', 'india_inflation_lag1', 'current_account_lag1', 'gdp_growth_roll3_std', 'gdp_growth_roll3_mean', 'forex_roll3_mean', 'inflation_roll3_mean', 'nepal_india_inflation_spread_lag1', 'forex_rapid_decline_lag1']


In [14]:
# Cell 13 — Variance Inflation Factor check
# VIF > 10 = severe multicollinearity, drop
# VIF 5-10 = moderate, flag for review
# VIF < 5 = acceptable

from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

X = final_df[feature_cols].values

vif_data = pd.DataFrame()
vif_data["feature"] = feature_cols
vif_data["VIF"] = [variance_inflation_factor(X, i) for i in range(X.shape[1])]
vif_data = vif_data.sort_values("VIF", ascending=False).reset_index(drop=True)

print(vif_data.to_string())

                              feature         VIF
0                      inflation_lag1         inf
1                india_inflation_lag1         inf
2   nepal_india_inflation_spread_lag1         inf
3                    forex_roll3_mean  113.250856
4               gdp_growth_roll3_mean   38.893268
5                inflation_roll3_mean   35.614933
6                gdp_growth_roll3_std   18.325009
7               india_gdp_growth_lag1   16.652246
8             remittance_pct_gdp_lag1   12.098464
9                     gdp_growth_lag1   10.775221
10               current_account_lag1    4.669821
11           forex_rapid_decline_lag1    2.790211


In [15]:
# Cell 14 — Drop high VIF features
# Drop inflation_lag1 and india_inflation_lag1 — perfectly collinear
#   with nepal_india_inflation_spread_lag1 (spread = nepal - india)
# Drop forex_roll3_mean — VIF 113, trend captured by forex_rapid_decline_lag1
# Drop gdp_growth_roll3_mean — VIF 38, collinear with gdp_growth_lag1
# Drop inflation_roll3_mean — VIF 35, collinear with spread feature
# Keep gdp_growth_roll3_std — captures volatility, different from level features
# Keep india_gdp_growth_lag1 — strong crisis signal (-0.322), accept VIF 16
# Keep remittance_pct_gdp_lag1 — accept VIF 12, important Nepal-specific feature
# Keep gdp_growth_lag1 — accept VIF 10, direct GDP signal

drop_cols2 = [
    'inflation_lag1',
    'india_inflation_lag1',
    'forex_roll3_mean',
    'gdp_growth_roll3_mean',
    'inflation_roll3_mean'
]

final_df = final_df.drop(columns=drop_cols2)

feature_cols = [c for c in final_df.columns
                if c not in ['year', 'crisis', 'triggers_fired']]

print(f"Features remaining: {len(feature_cols)}")
print(feature_cols)

Features remaining: 7
['gdp_growth_lag1', 'remittance_pct_gdp_lag1', 'india_gdp_growth_lag1', 'current_account_lag1', 'gdp_growth_roll3_std', 'nepal_india_inflation_spread_lag1', 'forex_rapid_decline_lag1']


In [16]:
# Cell 15 — VIF recheck on final 7 features
X = final_df[feature_cols].values

vif_data = pd.DataFrame()
vif_data["feature"] = feature_cols
vif_data["VIF"] = [variance_inflation_factor(X, i) for i in range(X.shape[1])]
vif_data = vif_data.sort_values("VIF", ascending=False).reset_index(drop=True)

print(vif_data.to_string())

                             feature       VIF
0            remittance_pct_gdp_lag1  7.261544
1                    gdp_growth_lag1  6.675449
2              india_gdp_growth_lag1  5.843511
3               gdp_growth_roll3_std  4.660483
4               current_account_lag1  3.002081
5           forex_rapid_decline_lag1  2.572981
6  nepal_india_inflation_spread_lag1  1.426323


In [17]:
# Cell 16 — Save final clean feature set
# 7 features, 26 rows, 1998-2024, 7 crisis years
# All VIF < 10, no perfect collinearity, strict t-1 lag throughout

final_df.to_csv("../data/features_clean.csv", index=False)

print("Saved: ../data/features_clean.csv")
print(f"Shape: {final_df.shape}")
print(f"Features: {feature_cols}")
print(f"Crisis years: {final_df[final_df['crisis']==1]['year'].tolist()}")
print(f"Crisis rate: {final_df['crisis'].mean()*100:.1f}%")

Saved: ../data/features_clean.csv
Shape: (26, 10)
Features: ['gdp_growth_lag1', 'remittance_pct_gdp_lag1', 'india_gdp_growth_lag1', 'current_account_lag1', 'gdp_growth_roll3_std', 'nepal_india_inflation_spread_lag1', 'forex_rapid_decline_lag1']
Crisis years: [1998, 2008, 2009, 2010, 2011, 2016, 2021]
Crisis rate: 26.9%


In [18]:
# Cell 16 — Save final clean feature set
# 7 features, 26 rows, 1998-2024, 7 crisis years
# All VIF < 10, no perfect collinearity, strict t-1 lag throughout

final_df.to_csv("../data/features_clean.csv", index=False)

print("Saved: ../data/features_clean.csv")
print(f"Shape: {final_df.shape}")
print(f"Features: {feature_cols}")
print(f"Crisis years: {final_df[final_df['crisis']==1]['year'].tolist()}")
print(f"Crisis rate: {final_df['crisis'].mean()*100:.1f}%")

Saved: ../data/features_clean.csv
Shape: (26, 10)
Features: ['gdp_growth_lag1', 'remittance_pct_gdp_lag1', 'india_gdp_growth_lag1', 'current_account_lag1', 'gdp_growth_roll3_std', 'nepal_india_inflation_spread_lag1', 'forex_rapid_decline_lag1']
Crisis years: [1998, 2008, 2009, 2010, 2011, 2016, 2021]
Crisis rate: 26.9%
